# 🚨 NLP Project: Disaster Tweet Classification

**Dataset:** NLP Getting Started — Kaggle  
**Task:** Binary Classification — Is this tweet about a real disaster? (`1`) or not? (`0`)  

---

## Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | Data Loading & Inspection |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Text Preprocessing |
| 4 | Feature Extraction (TF-IDF) |
| 5 | Model Training (Logistic Regression, Naive Bayes) |
| 6 | Model Evaluation (Accuracy, F1, Confusion Matrix) |
| 7 | Hyperparameter Tuning (GridSearchCV) |
| 8 | Model Comparison & Prediction on New Tweets |

## 1. Imports & Setup

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score
)

# Download required NLTK data
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet',
                 'omw-1.4', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All imports successful ✓')

---
## 2. Data Loading & Inspection

We load the training dataset provided for this project.  
Each row is a single tweet with a binary label:
- `target = 1` → the tweet describes a **real disaster**  
- `target = 0` → the tweet is **not about a real disaster** (figurative language, unrelated topic, etc.)

In [ ]:
df = pd.read_csv('train.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
print('--- Column Types ---')
print(df.dtypes)
print()
print('--- Missing Values ---')
print(df.isnull().sum())
print()
print('--- Class Distribution ---')
print(df['target'].value_counts())
print(f'\nDisaster tweets    : {df["target"].sum()} ({df["target"].mean():.1%})')
print(f'Non-disaster tweets: {(df["target"]==0).sum()} ({1-df["target"].mean():.1%})')

---
## 3. Exploratory Data Analysis (EDA)

Before building any model, we explore the data visually to understand:
- How balanced are the classes?
- How long are the tweets in each class?
- What words are most common in each class?
- What hashtags and patterns appear?

### 3.1 Class Distribution

In [ ]:
counts = df['target'].value_counts()
labels = ['Non-Disaster (0)', 'Disaster (1)']
colors = ['#3498db', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(labels, [counts[0], counts[1]], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_ylabel('Number of Tweets')
axes[0].set_title('Class Distribution — Bar Chart')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(
    [counts[0], counts[1]],
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.03, 0.03)
)
axes[1].set_title('Class Distribution — Proportion')

plt.suptitle('Is the Dataset Balanced?', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_class_distribution.png', bbox_inches='tight')
plt.show()

### 3.2 Tweet Length Analysis

Do disaster tweets tend to be longer or shorter than non-disaster tweets?

In [ ]:
df['word_count']  = df['text'].apply(lambda x: len(x.split()))
df['char_count']  = df['text'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#3498db', 'Non-Disaster'), (1, '#e74c3c', 'Disaster')]:
    subset = df[df['target'] == label]
    axes[0].hist(subset['word_count'], bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[1].hist(subset['char_count'], bins=30, alpha=0.6, color=color, label=name, edgecolor='white')

axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution by Class')
axes[0].legend()

axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Character Count Distribution by Class')
axes[1].legend()

plt.suptitle('Tweet Length: Disaster vs Non-Disaster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_tweet_length.png', bbox_inches='tight')
plt.show()

print('Average word count per class:')
print(df.groupby('target')['word_count'].mean().rename({0: 'Non-Disaster', 1: 'Disaster'}))

### 3.3 Most Frequent Words by Class

Before preprocessing — what raw words dominate each class?

In [ ]:
def get_top_words(texts, n=20):
    all_words = ' '.join(texts).lower().split()
    return Counter(all_words).most_common(n)

pos_words = get_top_words(df[df['target'] == 1]['text'])
neg_words = get_top_words(df[df['target'] == 0]['text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, color, title in [
    (axes[0], pos_words, '#e74c3c', 'Top 20 Words — Disaster Tweets'),
    (axes[1], neg_words, '#3498db', 'Top 20 Words — Non-Disaster Tweets')
]:
    words, counts = zip(*data)
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Frequency')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Most Frequent Words Before Preprocessing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_top_words_raw.png', bbox_inches='tight')
plt.show()

### 3.4 Top Hashtags by Class

Hashtags are strong signals in Twitter data — let's see which ones are most common in each class.

In [ ]:
def extract_hashtags(texts):
    hashtags = []
    for text in texts:
        hashtags.extend(re.findall(r'#\w+', text.lower()))
    return Counter(hashtags).most_common(15)

disaster_tags    = extract_hashtags(df[df['target'] == 1]['text'])
nondisaster_tags = extract_hashtags(df[df['target'] == 0]['text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, color, title in [
    (axes[0], disaster_tags,    '#e74c3c', 'Top Hashtags — Disaster'),
    (axes[1], nondisaster_tags, '#3498db', 'Top Hashtags — Non-Disaster')
]:
    if data:
        tags, counts = zip(*data)
        ax.barh(tags[::-1], counts[::-1], color=color, alpha=0.8, edgecolor='white')
        ax.set_xlabel('Frequency')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Hashtag Analysis by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_hashtags.png', bbox_inches='tight')
plt.show()

---
## 4. Text Preprocessing

Tweets are noisy — they contain URLs, @mentions, hashtags, emojis, and informal spelling.  
We apply a pipeline to clean and normalise the text before modelling:

| Step | Why? |
|------|------|
| Remove URLs | Not semantically meaningful for classification |
| Remove @mentions | Usernames carry no disaster signal |
| Strip # symbol | Keep the word, lose the symbol |
| Lowercase | `Fire` and `fire` are the same word |
| Remove numbers & punctuation | Reduce noise |
| Tokenise | Split into individual word tokens |
| Remove stopwords | `the`, `is`, `a` add noise, not signal |
| POS-aware Lemmatisation | `running → run`, `fires → fire` (context-aware) |

### Why lemmatisation over stemming?
Stemming cuts word endings using rules (`easily → easili`) — fast but crude.  
Lemmatisation uses vocabulary knowledge to return real dictionary words (`easily → easily`, `better → good`).  
For short texts like tweets, vocabulary quality matters more than speed.

In [ ]:
STOP_WORDS  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def get_wordnet_pos(tag):
    """Map Penn Treebank POS tags to WordNet POS tags for lemmatisation."""
    if tag.startswith('J'): return 'a'   # adjective
    elif tag.startswith('V'): return 'v' # verb
    elif tag.startswith('N'): return 'n' # noun
    elif tag.startswith('R'): return 'r' # adverb
    else: return 'n'

def clean_tweet(text):
    """Full preprocessing pipeline for a single tweet."""
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+', '', text)                   # remove @mentions
    text = re.sub(r'#', '', text)                      # strip # symbol, keep word
    text = text.lower()                                # lowercase
    text = re.sub(r'\d+', '', text)                    # remove digits
    text = re.sub(r'[^\w\s]', '', text)               # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()           # normalise whitespace

    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word not in STOP_WORDS and len(word) > 1
    ]
    return ' '.join(tokens)

# Apply to whole dataset
print('Preprocessing tweets...')
df['clean_text'] = df['text'].apply(clean_tweet)
print('Done ✓')

# Show before / after
examples = df[['text', 'clean_text', 'target']].sample(5, random_state=42)
pd.set_option('display.max_colwidth', 120)
examples

### 4.1 Top Words After Preprocessing

After removing stopwords and noise, what words now carry the most signal?

In [ ]:
clean_pos = get_top_words(df[df['target'] == 1]['clean_text'], n=20)
clean_neg = get_top_words(df[df['target'] == 0]['clean_text'], n=20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, color, title in [
    (axes[0], clean_pos, '#e74c3c', 'Top 20 Words — Disaster (after preprocessing)'),
    (axes[1], clean_neg, '#3498db', 'Top 20 Words — Non-Disaster (after preprocessing)')
]:
    words, counts = zip(*data)
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Frequency')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Most Frequent Words After Preprocessing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_top_words_clean.png', bbox_inches='tight')
plt.show()

---
## 5. Feature Extraction — TF-IDF

Machine learning models cannot work with raw text — we need to convert words into numbers.

### Bag of Words (BoW) vs TF-IDF

| Method | What it does | Weakness |
|--------|-------------|----------|
| **Bag of Words** | Counts raw word frequency | Common words like `fire` dominate even if they appear in every tweet |
| **TF-IDF** | Weights words by how *uniquely* informative they are | Slightly slower to compute |

**TF-IDF formula:**  
`TF-IDF(t, d) = TF(t, d) × log(N / df(t))`  
- `TF(t, d)` = how often term `t` appears in document `d`  
- `df(t)` = number of documents containing `t`  
- `N` = total documents  

Words that appear in almost every tweet (like `fire`) get **down-weighted**.  
Words that are specific to a few tweets get **up-weighted**.

We also use **bigrams** (`ngram_range=(1,2)`) to capture two-word phrases like `"not safe"` or `"flash flood"`.

In [ ]:
X = df['clean_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'\nClass balance in training set:')
print(y_train.value_counts(normalize=True).rename({0: 'Non-Disaster', 1: 'Disaster'}).round(3))

---
## 6. Model Training & Evaluation

We wrap each model in a **sklearn Pipeline** — this ensures that vectorisation is fitted on training data only, preventing data leakage.  

### Models we compare:

| Model | Why? |
|-------|------|
| **Logistic Regression** | Strong, interpretable baseline for binary text classification |
| **Multinomial Naive Bayes** | Classic probabilistic NLP model; fast, works well with count-based features |

In [ ]:
def evaluate_model(name, y_true, y_pred, ax):
    """Print classification report and plot confusion matrix on given axis."""
    print(f'\n{'='*50}')
    print(f'{name}')
    print(f'{'='*50}')
    print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'F1-Score : {f1_score(y_true, y_pred):.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=['Non-Disaster', 'Disaster']))

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Non-Disaster', 'Disaster'],
        yticklabels=['Non-Disaster', 'Disaster']
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix\n{name}')

### 6.1 Logistic Regression + TF-IDF

In [ ]:
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 4))
evaluate_model('Logistic Regression (TF-IDF, Unigrams+Bigrams)', y_test, y_pred_lr, ax)
plt.tight_layout()
plt.savefig('plot_cm_lr.png', bbox_inches='tight')
plt.show()

### 6.2 Multinomial Naive Bayes + CountVectorizer

In [ ]:
nb_pipe = Pipeline([
    ('bow', CountVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

nb_pipe.fit(X_train, y_train)
y_pred_nb = nb_pipe.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 4))
evaluate_model('Multinomial Naive Bayes (CountVectorizer, Unigrams+Bigrams)', y_test, y_pred_nb, ax)
plt.tight_layout()
plt.savefig('plot_cm_nb.png', bbox_inches='tight')
plt.show()

### 6.3 Cross-Validation

A single train/test split can be sensitive to which samples ended up in each split.  
**5-fold cross-validation** gives a more robust estimate of model performance by averaging results across 5 different splits.

In [ ]:
for name, pipe in [('Logistic Regression', lr_pipe), ('Naive Bayes', nb_pipe)]:
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    print(f'{name}')
    print(f'  F1 per fold : {[f"{s:.4f}" for s in scores]}')
    print(f'  Mean F1     : {scores.mean():.4f} ± {scores.std():.4f}')
    print()

---
## 7. Hyperparameter Tuning — GridSearchCV

Instead of guessing the best settings, **GridSearchCV** systematically tests all combinations and uses **3-fold cross-validation** to find which configuration generalises best.

We tune:
| Parameter | What it controls |
|-----------|------------------|
| `tfidf__max_features` | Vocabulary size (how many unique terms to keep) |
| `tfidf__ngram_range` | Unigrams only vs unigrams + bigrams |
| `clf__C` | Regularisation strength — lower C = stronger regularisation (less overfitting) |

In [ ]:
grid_pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__max_features': [3000, 5000, 10000],
    'tfidf__ngram_range' : [(1, 1), (1, 2)],
    'clf__C'             : [0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=grid_pipe,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('\nBest parameters:')
for k, v in grid.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV F1-Score: {grid.best_score_:.4f}')

In [ ]:
y_pred_tuned = grid.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 4))
evaluate_model('Tuned Logistic Regression (GridSearchCV)', y_test, y_pred_tuned, ax)
plt.tight_layout()
plt.savefig('plot_cm_tuned.png', bbox_inches='tight')
plt.show()

---
## 8. Model Comparison Summary

In [ ]:
results = {
    'Logistic Regression (baseline)' : (accuracy_score(y_test, y_pred_lr),   f1_score(y_test, y_pred_lr)),
    'Naive Bayes (baseline)'         : (accuracy_score(y_test, y_pred_nb),   f1_score(y_test, y_pred_nb)),
    'Logistic Regression (tuned)'    : (accuracy_score(y_test, y_pred_tuned),f1_score(y_test, y_pred_tuned)),
}

res_df = pd.DataFrame(
    [(name, acc, f1) for name, (acc, f1) in results.items()],
    columns=['Model', 'Accuracy', 'F1-Score']
).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print(res_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, color in [(axes[0], 'Accuracy', '#3498db'), (axes[1], 'F1-Score', '#e74c3c')]:
    bars = ax.bar(res_df['Model'], res_df[metric], color=color, alpha=0.85, edgecolor='black')
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel(metric)
    ax.set_title(f'Model Comparison — {metric}', fontweight='bold')
    ax.set_xticklabels(res_df['Model'], rotation=15, ha='right')
    for bar, val in zip(bars, res_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison: Accuracy & F1-Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight')
plt.show()

### 8.1 Error Analysis — What Does the Model Get Wrong?

Understanding **where the model fails** is just as important as knowing its accuracy.

In [ ]:
test_df = X_test.to_frame().copy()
test_df['actual']    = y_test.values
test_df['predicted'] = y_pred_tuned
test_df['original']  = df.loc[X_test.index, 'text'].values

# False Positives: model said disaster, actually not
false_pos = test_df[(test_df['actual'] == 0) & (test_df['predicted'] == 1)]
# False Negatives: model said not disaster, actually is
false_neg = test_df[(test_df['actual'] == 1) & (test_df['predicted'] == 0)]

print(f'False Positives (predicted disaster, actually not): {len(false_pos)}')
print(f'False Negatives (predicted not disaster, actually is): {len(false_neg)}')

print('\n--- Sample False Positives (figurative/metaphorical language) ---')
for tweet in false_pos['original'].head(5).values:
    print(f'  → "{tweet}"')

print('\n--- Sample False Negatives (real disaster, model missed) ---')
for tweet in false_neg['original'].head(5).values:
    print(f'  → "{tweet}"')

---
## 9. Predict on New Tweets

A utility function that applies the same preprocessing pipeline and uses our best model.

In [ ]:
def predict_tweet(tweet, model=grid):
    cleaned    = clean_tweet(tweet)
    prediction = model.predict([cleaned])[0]
    proba      = model.predict_proba([cleaned])[0]
    label      = '🚨 DISASTER' if prediction == 1 else '✅ NOT DISASTER'
    confidence = proba[prediction]
    print(f'Tweet      : "{tweet}"')
    print(f'Prediction : {label}  (confidence: {confidence:.1%})')
    print()

test_tweets = [
    "BREAKING: Massive earthquake hits the coast, tsunami warning issued for the region",
    "This math homework is absolutely destroying me 😭💀",
    "Evacuations underway as wildfire spreads across 3,000 acres in California",
    "My new shoes are on fire, literally the best purchase I've ever made",
    "Flash flood warning issued for downtown area — residents urged to evacuate immediately"
]

for tweet in test_tweets:
    predict_tweet(tweet)

---
## 10. Summary & Reflections

| Stage | Decision Made | Justification |
|-------|--------------|---------------|
| **Preprocessing** | Remove URLs, @mentions, #symbols + lemmatise | Reduce noise specific to Twitter format |
| **Feature Extraction** | TF-IDF with unigrams + bigrams | Down-weights common words; bigrams capture phrases like `flash flood` |
| **Model** | Logistic Regression | Strong baseline for binary classification; interpretable; fast |
| **Evaluation** | F1-score as primary metric | Dataset is slightly imbalanced — F1 balances precision and recall |
| **Tuning** | GridSearchCV (3-fold CV) | Systematic hyperparameter search; avoids overfitting to test set |

### Key Findings
- Logistic Regression with TF-IDF is a **strong, efficient baseline** for this problem
- The biggest challenge is **figurative language**: tweets like *"this movie is a disaster"* or *"I'm dying of boredom"* confuse the model
- Bigrams modestly improve performance by capturing meaningful two-word phrases
- Cross-validation confirms the model generalises well across different data splits

### Limitations
- **No semantic understanding**: TF-IDF treats words as independent — it cannot understand context or sarcasm
- **Twitter-specific noise**: Even after cleaning, informal spelling and slang remain challenging
- **Class imbalance**: Non-disaster tweets outnumber disaster tweets (~57% vs ~43%), which can bias predictions
- **Potential improvement**: Transformer-based models (BERT, RoBERTa) would handle context and figurative language far better